In [ ]:
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
from galfits import images as IM
import pickle
from astropy.stats import sigma_clipped_stats
from astropy.visualization.mpl_normalize import simple_norm
from astropy.table import Table, Column
import matplotlib.pyplot as plt
from galfits import gsutils
from galfits import sed_interp as SI
from galfits import galaxy as GAL
import numpy as np
import time
import jax.numpy as jnp
import wget,shutil
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM,z_at_value
from scipy.ndimage import zoom
from astropy.io import fits, ascii
from matplotlib import patches 
from astropy.wcs import WCS
from skimage.transform import resize
from astropy.visualization import make_lupton_rgb
plt.style.use("classic")
plt.rc('font',family='Times New Roman')


In [ ]:
priorpath = 
config_file_path=
workplace = 

Myfitter,targ,fs = gsutils.read_config_file(config_file_path,workplace,priorpath=priorpath)
name = targ
smfile = ascii.read(workplace+'{0}.gssummary'.format(name))
for loopx in range(len(smfile)):
    Myfitter.lmParameters[smfile['pname'][loopx]].set(value=smfile['best_value'][loopx], min=smfile['best_value'][loopx]-0.1,max=smfile['best_value'][loopx]+0.1)
Myfitter.loose_fix_pars()

In [ ]:
images = []
models = []
agns = []
galaxies = []
gmodel = []
residuals = []
bands = []
MASK = []
pltwave, Sedcomp, Sedlabel, z0 = Myfitter.cal_model_image()
nimage = 3
for i in range(nimage):
    im = Myfitter.GSdata.get_image(i)
    sky_mean, sky_median, sky_std = sigma_clipped_stats(im.cut_image, sigma=3.0, maxiters=5)
    im.cut_image = im.cut_image - sky_median
    images.append(im.cut_image)
    models.append(im.model_image)
    residuals.append((im.cut_image-im.model_image)/im.cut_sigma_image)
      

## plot subcomponents

In [ ]:
# you can manipulate the following variables to plot different components
galaxy_model = Myfitter.gmodel_list[0]
print (galaxy_model.name) # you can check the name of the component
# also you can check the component names
print (galaxy_model.subnames)
# if you want to mask some components, say 'bulge' you can do like this
print (galaxy_model.sedmodelist['bulge'])
# to mask it
galaxy_model.sedmodelist['bulge'] = jnp.array([0.,0.,0.],dtype=jnp.float32)
# then re-run the plot
pltwave, Sedcomp, Sedlabel, z0 = Myfitter.cal_model_image()